In [1]:
!pip install wurlitzer
from wurlitzer import sys_pipes_forever

sys_pipes_forever()

In [2]:
#using uproot to write generated event into TTree

!pip install uproot
import uproot

import numpy as np
import pandas as pd



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.5/397.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924.2 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.8/657.8 kB 32.9 MB/s eta 0:00:00


In [3]:
#read event data

file4 = uproot.open("PythiaEventsBatchTest.root")

tree = file4['pdEventTree']

#read branches into Awkward array, then numpy arrays

E_array = tree['E'].array().to_numpy()
px_array = tree['px'].array().to_numpy()
py_array = tree['py'].array().to_numpy()
pz_array = tree['pz'].array().to_numpy()
eta_array = tree['eta'].array().to_numpy()
phi_array = tree['phi'].array().to_numpy()



In [4]:
# using FastJet now:
!pip install fastjet
import fastjet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.7/182.7 kB 8.4 MB/s eta 0:00:00


In [5]:
#Bringing in FastJet's anti-kT, creating PseudoJets from input data
#Can make minor edits to use CA, kT algorithms as well

antikt_jetdef = fastjet.JetDefinition(fastjet.antikt_algorithm, 0.4)

kt_jetdef = fastjet.JetDefinition(fastjet.kt_algorithm, 0.4)

ca_jetdef = fastjet.JetDefinition(fastjet.cambridge_algorithm, 0.4)

jet_set = [fastjet.PseudoJet(px_array[i], py_array[i], pz_array[i], E_array[i]) for i in range(len(E_array))]



In [6]:
for i,j in enumerate(jet_set):
    j.set_user_index(i)

In [7]:
#Clustering

antikTcluster = fastjet.ClusterSequence(jet_set, antikt_jetdef)

ktcluster = fastjet.ClusterSequence(jet_set, kt_jetdef)

cacluster = fastjet.ClusterSequence(jet_set, ca_jetdef)



#--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optionally PLB641(2006)57 [hep-ph/0512210].   
#                                                                       
# FastJet is provided without warranty under the GNU GPL v2 or higher.  
# It uses T. Chan's closest pair algorithm, S. Fortune's Voronoi code,
# CGAL and 3rd party plugin jet algorithms. See COPYING file for details.
#--------------------------------------------------------------------------


In [8]:
#looking into output

antikTcluster

<fastjet._swig.ClusterSequence; proxy of <Swig Object of type 'fastjet::ClusterSequence *' at 0x7d1ebc057df0> >

In [9]:
antikt_clustered_jets = antikTcluster.inclusive_jets()

In [10]:
#writing output final jets to data file:

kt_clustered_jets = ktcluster.inclusive_jets()

ca_clustered_jets = cacluster.inclusive_jets()

antikt_jet_constituents = [jet.constituents() for jet in antikt_clustered_jets]

kt_jet_constituents = [jet.constituents() for jet in ktcluster.inclusive_jets()]

ca_jet_constituents = [jet.constituents() for jet in cacluster.inclusive_jets()]


def write_file(jet_constituents, clustered_jets, jet_type):
  jet_ids = []

  for j_c in jet_constituents:
      jet_ids.append([j.user_index() for j in j_c])

  file_out = uproot.recreate(f"FJ_{jet_type}.root")

  output_dict = {
      "E": [jet.E() for jet in clustered_jets],
      "px": [jet.px() for jet in clustered_jets],
      "py": [jet.py() for jet in clustered_jets],
      "pz": [jet.pz() for jet in clustered_jets],
      "id": jet_ids,
      "pT": [jet.pt() for jet in clustered_jets],
      "eta": [jet.eta() for jet in clustered_jets],
      "phi": [jet.phi() for jet in clustered_jets]
  }

  output_df = pd.DataFrame(output_dict)

  file_out.mktree("finalJets", output_df)

  file_out["finalJets"]

write_file(antikt_jet_constituents, antikt_clustered_jets,"antikt")
write_file(kt_jet_constituents, kt_clustered_jets,"kt")
write_file(ca_jet_constituents, ca_clustered_jets,"CA")





In [11]:
new_file_in = uproot.recreate("PythiaEventsBatchTest_Plot.root")

input_dict = {
    "E": E_array,
    "px": px_array,
    "py": py_array,
    "pz": pz_array,
    "pT": [np.sqrt(px_array[i]**2 + py_array[i]**2) for i in range(len(E_array))],
    "eta": eta_array,
    "phi": phi_array
}

file_in_df = pd.DataFrame(input_dict)

new_file_in.mktree("pdEventTree", file_in_df)

new_file_in["pdEventTree"]

<WritableTree '/pdEventTree' at 0x7d1eb7e4ba40>

In [25]:
########## Additional miscellaneous; trying to look under hood of FastJet ##########

In [12]:
attrs = [attr for attr in dir(antikt_clustered_jets[0]) if not attr.startswith('__')]

In [13]:
print(attrs)

['E', 'Et', 'Et2', 'NUM_COORDINATES', 'SIZE', 'T', 'X', 'Y', 'Z', 'area', 'area_4vector', 'area_error', 'associated_cluster_sequence', 'associated_cs', 'beam_distance', 'boost', 'cluster_hist_index', 'cluster_sequence_history_index', 'constituents', 'contains', 'cos_theta', 'delta_R', 'delta_phi_to', 'description', 'e', 'eta', 'exclusive_subdmerge', 'exclusive_subdmerge_max', 'exclusive_subjets', 'exclusive_subjets_up_to', 'four_mom', 'has_area', 'has_associated_cluster_sequence', 'has_associated_cs', 'has_child', 'has_constituents', 'has_exclusive_subjets', 'has_parents', 'has_partner', 'has_pieces', 'has_structure', 'has_user_info', 'has_valid_cluster_sequence', 'has_valid_cs', 'is_inside', 'is_pure_ghost', 'kt2', 'kt_distance', 'm', 'm2', 'modp', 'modp2', 'mperp', 'mperp2', 'mt', 'mt2', 'n_exclusive_subjets', 'perp', 'perp2', 'phi', 'phi_02pi', 'phi_std', 'pieces', 'plain_distance', 'pseudorapidity', 'pt', 'pt2', 'px', 'py', 'python_info', 'pz', 'rap', 'rapidity', 'reset', 'reset_Pt

In [14]:
import inspect

In [15]:
for name, member in inspect.getmembers(antikt_clustered_jets[0]):
    if not name.startswith('__'):
      print(name)

E
Et
Et2
NUM_COORDINATES
SIZE
T
X
Y
Z
area
area_4vector
area_error
associated_cluster_sequence
associated_cs
beam_distance
boost
cluster_hist_index
cluster_sequence_history_index
constituents
contains
cos_theta
delta_R
delta_phi_to
description
e
eta
exclusive_subdmerge
exclusive_subdmerge_max
exclusive_subjets
exclusive_subjets_up_to
four_mom
has_area
has_associated_cluster_sequence
has_associated_cs
has_child
has_constituents
has_exclusive_subjets
has_parents
has_partner
has_pieces
has_structure
has_user_info
has_valid_cluster_sequence
has_valid_cs
is_inside
is_pure_ghost
kt2
kt_distance
m
m2
modp
modp2
mperp
mperp2
mt
mt2
n_exclusive_subjets
perp
perp2
phi
phi_02pi
phi_std
pieces
plain_distance
pseudorapidity
pt
pt2
px
py
python_info
pz
rap
rapidity
reset
reset_PtYPhiM
reset_momentum
reset_momentum_PtYPhiM
set_cached_rap_phi
set_cluster_hist_index
set_cluster_sequence_history_index
set_python_info
set_structure_shared_ptr
set_user_index
set_user_info
set_user_info_shared_ptr
squared_

In [16]:
for attr_name in attrs:
    member = getattr(antikt_clustered_jets[0], attr_name)
    doc = inspect.getdoc(member)
    if doc:
        print(f"--- Help for <{attr_name}> ---")
        print(doc)
        print("\n")
    else:
        print(f"--- (!!!) No help message found for {attr_name} ---")
        print("\n")

--- Help for <E> ---
`E() const -> double`  


--- Help for <Et> ---
`Et() const -> double`  

return the transverse energy  


--- Help for <Et2> ---
`Et2() const -> double`  

return the transverse energy squared  


--- Help for <NUM_COORDINATES> ---
int([x]) -> integer
int(x, base=10) -> integer

Convert a number or string to an integer, or return 0 if no arguments
are given.  If x is a number, return x.__int__().  For floating-point
numbers, this truncates towards zero.

If x is not a number or if base is given, then x must be a string,
bytes, or bytearray instance representing an integer literal in the
given base.  The literal can be preceded by '+' or '-' and be surrounded
by whitespace.  The base defaults to 10.  Valid bases are 0 and 2-36.
Base 0 means to interpret the base from the string as an integer literal.
>>> int('0b100', base=0)
4


--- Help for <SIZE> ---
int([x]) -> integer
int(x, base=10) -> integer

Convert a number or string to an integer, or return 0 if no argume

In [17]:
import inspect

print("--- Help messages for methods of clustered_jets[0] ---\n")

for name, member in inspect.getmembers(antikt_clustered_jets[0]):
    if not name.startswith('__') and inspect.ismethod(member):
        doc = inspect.getdoc(member)
        if doc:
            print(f"--- Help for <{name}> ---")
            print(doc)
            print("\n")
        else:
            print(f"--- (!!!) No help message found for method {name} ---")
            print("\n")

--- Help messages for methods of clustered_jets[0] ---

--- Help for <E> ---
`E() const -> double`  


--- Help for <Et> ---
`Et() const -> double`  

return the transverse energy  


--- Help for <Et2> ---
`Et2() const -> double`  

return the transverse energy squared  


--- Help for <area> ---
`area() const -> double`  

return the jet (scalar) area.  

throws an Error if there is no support for area in the parent CS  


--- Help for <area_4vector> ---
`area_4vector() const -> PseudoJet`  

return the jet 4-vector area.  

throws an Error if there is no support for area in the parent CS  


--- Help for <area_error> ---
`area_error() const -> double`  

return the error (uncertainty) associated with the determination of the area of
this jet.  

throws an Error if there is no support for area in the parent CS  


--- Help for <associated_cluster_sequence> ---
`associated_cluster_sequence() const -> const ClusterSequence *`  

get a (const) pointer to the parent ClusterSequence (NULL

In [18]:
# To get the cluster history index of a specific jet, call it on a PseudoJet object.
# For example, for the first clustered jet:
antikt_hist_index_of_first_jet = antikt_clustered_jets[0].cluster_hist_index()

In [19]:
antikt_hist_index_of_first_jet

136

In [20]:
cluster_sequence_obj = antikt_clustered_jets[0].associated_cluster_sequence()
print(cluster_sequence_obj)


<fastjet._swig.ClusterSequence; proxy of <Swig Object of type 'fastjet::ClusterSequence *' at 0x7d1eb6ea96b0> >


In [21]:
cluster_sequence_obj_history = cluster_sequence_obj.history()
print(cluster_sequence_obj_history)

<Swig Object of type 'std::vector< fastjet::ClusterSequence::history_element,std::allocator< fastjet::ClusterSequence::history_element > > *' at 0x7d1eb6ed7870>


In [22]:
cluster_sequence_obj_history.size()

AttributeError: 'swig_runtime_data5.SwigPyObject' object has no attribute 'size'

In [23]:
for i in range(cluster_sequence_obj_history.size()):
    print(cluster_sequence_obj_history.at(i))

AttributeError: 'swig_runtime_data5.SwigPyObject' object has no attribute 'size'

In [24]:
for history_elem in cluster_sequence_obj_history:
    print(history_elem)

TypeError: 'swig_runtime_data5.SwigPyObject' object is not iterable